# 04 — Comparison with Reference Data and Reduced Observables

This notebook is the most comparative of the four.  
Its aim is to demonstrate how toy-model outputs can be turned into **reduced observables** and compared against a reference signal.

## Learning goals

- define reduced observables suitable for comparison,
- compare two signals in time and in the frequency domain,
- understand the role of preprocessing and normalization,
- connect notebook analysis to `models/data_driven_models/` and `diagnostics/`.


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

print("Working directory:", Path.cwd())
print("Repository root guessed as:", ROOT)

def repo_file_info(relpath):
    p = ROOT / relpath
    return {"exists": p.exists(), "size": p.stat().st_size if p.exists() else None, "path": str(p)}

def sign_changes(x):
    s = np.sign(x)
    s[s == 0] = np.nan
    valid = ~np.isnan(s)
    sv = s[valid]
    return int(np.sum(sv[1:] * sv[:-1] < 0))

def polarity_series(x):
    p = np.sign(x)
    if len(p) == 0:
        return p
    # carry last sign through zeros
    for i in range(1, len(p)):
        if p[i] == 0:
            p[i] = p[i-1]
    if p[0] == 0:
        nz = np.flatnonzero(p != 0)
        if len(nz):
            p[:nz[0]] = p[nz[0]]
    return p

def residence_times_from_signal(x, dt=1.0):
    p = polarity_series(np.asarray(x))
    if len(p) == 0:
        return np.array([])
    durations = []
    current = p[0]
    count = 1
    for val in p[1:]:
        if val == current:
            count += 1
        else:
            durations.append(count * dt)
            current = val
            count = 1
    durations.append(count * dt)
    return np.array(durations)

def power_spectrum(x, dt=1.0):
    x = np.asarray(x)
    x = x - np.mean(x)
    freqs = np.fft.rfftfreq(len(x), d=dt)
    spec = np.abs(np.fft.rfft(x))**2 / len(x)
    return freqs[1:], spec[1:]

rng = np.random.default_rng(42)


## 1. Repo-aligned philosophy

The folder `models/data_driven_models/` suggests that the repository intends to include reduced observables and comparison-oriented workflows.  
The public raw view of `reduced_observables.py` is currently empty, so this notebook provides a robust self-contained workflow that can later be migrated into that script.


In [ ]:

info = repo_file_info("models/data_driven_models/reduced_observables.py")
print(info)


## 2. A synthetic reference workflow

We generate:

- a synthetic toy-model signal \(x(t)\),
- a synthetic reference signal \(y(t)\),

and compare them through:

- normalization,
- correlation,
- polarity structure,
- spectra.


In [ ]:

def zscore(x):
    x = np.asarray(x)
    return (x - np.mean(x)) / np.std(x)

# toy-model-like signal
t = np.linspace(0, 200, 10000)
x = np.sin(0.12*t) + 0.35*np.sin(0.03*t + 0.4) + 0.25*rng.standard_normal(len(t))

# reference-like signal
y = 0.8*np.sin(0.12*t + 0.5) + 0.30*np.sin(0.028*t + 0.8) + 0.25*rng.standard_normal(len(t))

xz = zscore(x)
yz = zscore(y)

fig, ax = plt.subplots()
ax.plot(t, xz, label="toy-model observable", alpha=0.8)
ax.plot(t, yz, label="reference observable", alpha=0.8)
ax.set_title("Normalized observables for comparison")
ax.set_xlabel("time")
ax.set_ylabel("z-score")
ax.legend()
plt.show()


## 3. Correlation and lag analysis

A good comparison notebook should examine not only visual similarity, but also alignment and lag structure.


In [ ]:

corr = np.corrcoef(xz, yz)[0,1]
print("Pearson correlation:", corr)

cc = np.correlate(xz - xz.mean(), yz - yz.mean(), mode="full")
lags = np.arange(-len(xz)+1, len(xz))
lag_best = lags[np.argmax(cc)]
print("Best lag (in samples):", lag_best)

fig, ax = plt.subplots()
ax.plot(lags, cc)
ax.set_title("Cross-correlation function")
ax.set_xlabel("lag (samples)")
ax.set_ylabel("cross-correlation")
plt.show()


## 4. Spectral comparison

Reduced observables can be compared spectrally even when their time-domain trajectories are not identical.


In [ ]:

fx, Sx = power_spectrum(xz, dt=t[1]-t[0])
fy, Sy = power_spectrum(yz, dt=t[1]-t[0])

fig, ax = plt.subplots()
ax.loglog(fx, Sx, label="toy-model observable")
ax.loglog(fy, Sy, label="reference observable")
ax.set_title("Spectral comparison")
ax.set_xlabel("frequency")
ax.set_ylabel("power")
ax.legend()
plt.show()


## 5. Simple event-based comparison

We can also compare reduced event structure, for example sign-change counts or residence-time distributions.


In [ ]:

rx = residence_times_from_signal(xz, dt=t[1]-t[0])
ry = residence_times_from_signal(yz, dt=t[1]-t[0])

print("Toy-model sign changes:", sign_changes(xz))
print("Reference sign changes:", sign_changes(yz))
print("Mean residence time (toy-model):", rx.mean())
print("Mean residence time (reference):", ry.mean())

fig, ax = plt.subplots()
ax.hist(rx, bins=30, alpha=0.6, label="toy-model")
ax.hist(ry, bins=30, alpha=0.6, label="reference")
ax.set_title("Residence-time comparison")
ax.set_xlabel("duration")
ax.set_ylabel("count")
ax.legend()
plt.show()


## 6. Interpretation

This notebook illustrates a general principle:

A toy model becomes scientifically useful not only when it generates a trajectory, but when it generates **observables that can be compared meaningfully**.

That is the conceptual role of the repository’s `data_driven_models` folder.


## 7. Connection to the repository

This notebook is naturally linked to:

- `models/data_driven_models/reduced_observables.py`
- `diagnostics/power_spectra.py`
- `diagnostics/polarity.py`
- `diagnostics/reversal_statistics.py`

It can later be upgraded by replacing the synthetic reference signal with:

- processed paleomagnetic benchmark data,
- outputs from the domino model,
- outputs from the bistable model,
- reduced observables from larger numerical simulations.


## 8. Suggested exercises

1. Replace the synthetic reference by a CSV file from `data/processed/` once available.  
2. Compare two different toy-model classes spectrally.  
3. Add smoothing and investigate how preprocessing changes the result.  
4. Define a custom distance metric between observables.
